# Transfer Entropy Between Zone Prices

Does a zone's price history help predict another zone's future price, beyond what that
zone's own past already tells you? That's transfer entropy: a directed,
information-theoretic measure, distinct from the physical-flow network built in
[network-analysis](../network-analysis/network_analysis.ipynb).

One caveat up front, repeated later next to the claim it bears on: transfer entropy is a
statement about predictability, not mechanism. A high TE(A &rarr; B) means A's past prices
reduce uncertainty about B's future price beyond B's own history &ndash; it does not mean A
*causes* B's price to move (see Candelaria et al., 2025, [arXiv:2508.08513](https://arxiv.org/pdf/2508.08513),
who make the same point working the NY grid).

This is phase 1: a hand-rolled, single-lag, binned transfer entropy estimator, no
significance testing yet &ndash; just establishing whether the information-flow graph it
produces roughly agrees or disagrees with the physical-flow graph. Lag sweeps, surrogate
significance testing, and a cross-check against `ci-flows`'s counter-intuitive-flow events
are deferred to a later pass.

In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
from common import drop_ukraine, fetch_flows_range, fetch_prices_range, save_chart

## Same window and physical graph as network-analysis

Reusing [network-analysis](../network-analysis/network_analysis.ipynb)'s window and
flow-weighted graph construction so the transfer-entropy comparison is apples-to-apples
against an already-reviewed baseline, not a freshly re-derived one.

In [ ]:
date_from = "2025-06-10"
date_to = "2026-06-10"

flows = fetch_flows_range(date_from, date_to)
print(f"{len(flows):,} observations, {flows['border'].nunique()} borders, {date_from} to {date_to}")

In [ ]:
def build_flow_graph(flows: pd.DataFrame) -> nx.DiGraph:
    flows = drop_ukraine(flows)
    G = nx.DiGraph()
    for (zone_from, zone_to), sub in flows.groupby(["bidding_zone_from", "bidding_zone_to"]):
        forward = sub.loc[sub["net_flow_mw"] > 0, "net_flow_mw"].sum()
        backward = -sub.loc[sub["net_flow_mw"] < 0, "net_flow_mw"].sum()
        if forward > 0:
            G.add_edge(zone_from, zone_to, weight=forward)
        if backward > 0:
            G.add_edge(zone_to, zone_from, weight=backward)
    return G


G_flow = build_flow_graph(flows)
zones = sorted(G_flow.nodes())
print(f"{len(zones)} zones, {G_flow.number_of_edges()} directed edges")

## Pulling per-zone day-ahead prices

Native 15-minute resolution, genuine per-zone series from `/api/v1/prices` (not the
flows endpoint's `price_delta`, which conflates both zones of a border into one number).
Same zone set as the physical graph, same window.

In [ ]:
prices = fetch_prices_range(date_from, date_to, zone=zones)
# chunked daily fetches overlap at day boundaries, which pivot can't reshape
prices = prices.drop_duplicates(["datetime", "bidding_zone"])
price_wide = prices.pivot(index="datetime", columns="bidding_zone", values="price_mwh").sort_index()
# isolated 15-min gaps (a handful of bars per zone, mostly DST transitions)
price_wide = price_wide.ffill().bfill()
price_wide = price_wide.dropna(axis=1, how="any")  # MK/AL: real multi-day gaps, not isolated bars
dropped = set(zones) - set(price_wide.columns)
if dropped:
    print(f"dropped {len(dropped)} zones with gaps in native-resolution price data: {sorted(dropped)}")
zones = sorted(price_wide.columns)
print(f"{len(price_wide):,} timestamps, {len(zones)} zones with complete price series")

## Hand-rolled binned transfer entropy

TE(X &rarr; Y) at lag 1: how much does knowing X's previous value reduce uncertainty about
Y's current value, beyond what Y's own previous value already gives you.

$$TE_{X \to Y} = \sum p(y_t, y_{t-1}, x_{t-1}) \log_2 \frac{p(y_t \mid y_{t-1}, x_{t-1})}{p(y_t \mid y_{t-1})}$$

Prices are continuous, so each series is discretized into `n_bins` equal-frequency bins
first (a quantile cut, not equal-width &ndash; keeps every bin populated regardless of a
price series' skew). 8 bins is a simplification, not a tuned choice: phase 2 is where a
lag/bin-count sweep would show whether the ranking below is robust to that choice.

In [ ]:
def transfer_entropy(x: pd.Series, y: pd.Series, n_bins: int = 8, lag: int = 1) -> float:
    """Binned TE(X -> Y). Discretizes once, then delegates to the numpy core so the
    same binned arrays can be reused directly by the surrogate test below without
    re-running qcut on every shuffle."""
    x_bin = pd.qcut(x, n_bins, labels=False, duplicates="drop").to_numpy()
    y_bin = pd.qcut(y, n_bins, labels=False, duplicates="drop").to_numpy()
    return transfer_entropy_from_bins(x_bin, y_bin, lag)


def transfer_entropy_from_bins(x_bin: np.ndarray, y_bin: np.ndarray, lag: int = 1) -> float:
    """Same estimator as `transfer_entropy`, operating on already-discretized arrays.
    Joint/marginal counts via np.bincount on flattened multi-index codes instead of a
    pandas groupby+merge - the groupby version was fine for a one-off pairwise sweep,
    but the surrogate test below calls this ~30,000 times and needs it fast."""
    bx, by = x_bin.max() + 1, y_bin.max() + 1
    y_t, y_lag, x_lag = y_bin[lag:], y_bin[:-lag], x_bin[:-lag]
    n = len(y_t)

    c_joint = np.bincount((y_t * by + y_lag) * bx + x_lag, minlength=by * by * bx).astype(float)
    c_ylag_xlag = np.bincount(y_lag * bx + x_lag, minlength=by * bx).astype(float)
    c_yt_ylag = np.bincount(y_t * by + y_lag, minlength=by * by).astype(float)
    c_ylag = np.bincount(y_lag, minlength=by).astype(float)

    flat_idx = np.flatnonzero(c_joint)
    yt_, rem = np.divmod(flat_idx, by * bx)
    ylag_, xlag_ = np.divmod(rem, bx)

    p_joint = c_joint[flat_idx] / n
    p_ylag_xlag = c_ylag_xlag[ylag_ * bx + xlag_] / n
    p_yt_ylag = c_yt_ylag[yt_ * by + ylag_] / n
    p_ylag = c_ylag[ylag_] / n
    return float(np.sum(p_joint * np.log2((p_joint / p_ylag_xlag) / (p_yt_ylag / p_ylag))))


# sanity check: two unrelated noise series have no real information flow between
# them, but a plug-in binned TE estimator is upward-biased on finite samples (bias
# scales roughly with n_bins^3 / n_observations) - it never actually reads 0. That
# bias is exactly why raw TE values below can't be compared to zero, only to a
# shuffled-null baseline (phase 2).
rng = np.random.default_rng(42)
noise_a = pd.Series(rng.normal(size=2000))
noise_b = pd.Series(rng.normal(size=2000))
print(f"TE(noise -> unrelated noise), finite-sample bias floor: {transfer_entropy(noise_a, noise_b):.4f}")

## Pairwise transfer entropy across all zones

Every ordered pair, not just physical neighbors &ndash; the point of this piece is whether
information flow lines up with physical flow, which requires looking beyond borders that
already exist in the flow graph.

In [ ]:
te_records = []
for source in zones:
    for target in zones:
        if source == target:
            continue
        te = transfer_entropy(price_wide[source], price_wide[target])
        te_records.append({"source": source, "target": target, "te": te})
te_df = pd.DataFrame(te_records)
te_df.sort_values("te", ascending=False).head(10)

## Price-setting zones: net TE out-strength

A zone with high *out*-TE and low *in*-TE is, on this measure, a price leader: its past
predicts others' futures more than the reverse. This ranking uses all pairs regardless of
physical topology &ndash; a zone can lead on price without a direct interconnector.

Caveat restated: this is "leads in predictability," not "causes." Candelaria et al. (2025)
flag the same risk of over-reading transfer entropy as causal on real grid data.

In [ ]:
out_strength = te_df.groupby("source")["te"].sum()
in_strength = te_df.groupby("target")["te"].sum()
net_te = (out_strength - in_strength).rename("net_te_out_strength").sort_values(ascending=False)

fig = px.bar(
    net_te.rename_axis("zone").reset_index().tail(10),
    x="net_te_out_strength",
    y="zone",
    orientation="h",
    title=f"Top 10 zones by net transfer-entropy out-strength, {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "net_te_out_strength_top10")

## Edge-level agreement with the physical-flow graph

For every border that actually exists in the physical graph, does the dominant TE
direction (whichever of TE(A &rarr; B), TE(B &rarr; A) is larger) match the dominant physical
flow direction (whichever of the two signed edge weights is larger)? Agreement is the
"price leader = physical exporter" baseline case; disagreement is the interesting case
(deferred cross-check: whether it clusters near `ci-flows`'s counter-intuitive events).

In [ ]:
zone_set = set(zones)
border_pairs = {tuple(sorted((u, v))) for u, v in G_flow.edges() if u in zone_set and v in zone_set}

agreement_records = []
for zone_a, zone_b in border_pairs:
    flow_ab = G_flow.get_edge_data(zone_a, zone_b, default={"weight": 0.0})["weight"]
    flow_ba = G_flow.get_edge_data(zone_b, zone_a, default={"weight": 0.0})["weight"]
    physical_direction = zone_a if flow_ab >= flow_ba else zone_b

    te_ab = te_df.loc[(te_df["source"] == zone_a) & (te_df["target"] == zone_b), "te"].iloc[0]
    te_ba = te_df.loc[(te_df["source"] == zone_b) & (te_df["target"] == zone_a), "te"].iloc[0]
    te_direction = zone_a if te_ab >= te_ba else zone_b

    agreement_records.append(
        {
            "border": f"{zone_a} - {zone_b}",
            "physical_exporter": physical_direction,
            "te_leader": te_direction,
            "agrees": physical_direction == te_direction,
        }
    )

agreement_df = pd.DataFrame(agreement_records)
agreement_rate = agreement_df["agrees"].mean()
print(f"{agreement_df['agrees'].sum()} / {len(agreement_df)} borders agree ({agreement_rate:.1%})")
agreement_df[~agreement_df["agrees"]]

## What phase 1 shows, and what it doesn't

The agreement rate above is the headline number for this pass: how often the zone with
the larger TE out-edge on a border is also the zone that physically exports more power
across it. Read it as a baseline, not a final answer &ndash; a single lag (15 minutes) and a
fixed 8-bin discretization, with no significance test separating a real TE edge from
estimator noise on this amount of data.

Phase 2 (deferred): a lag sweep to check the price-setting-zone ranking is stable, a
block-shuffle surrogate test for which TE edges are distinguishable from noise, and a
cross-check of disagreeing borders against `ci-flows`'s counter-intuitive-event dataset.

## Phase 2a: is the price-setting-zone ranking stable across lags?

Phase 1 used a single 15-minute lag. If `CH`/`RS` topping the net-TE-out-strength ranking
is a real pattern rather than an artifact of that one specific lag, the same zones should
stay near the top at 30 and 60 minutes too.

In [ ]:
lag_sweep_records = []
for lag_steps in (1, 2, 4):
    for source in zones:
        for target in zones:
            if source == target:
                continue
            te = transfer_entropy(price_wide[source], price_wide[target], lag=lag_steps)
            lag_sweep_records.append({"lag_steps": lag_steps, "source": source, "target": target, "te": te})

lag_sweep_df = pd.DataFrame(lag_sweep_records)
net_te_by_lag = (
    lag_sweep_df.groupby(["lag_steps", "source"])["te"].sum()
    - lag_sweep_df.groupby(["lag_steps", "target"])["te"].sum()
).rename("net_te_out_strength")

rank_by_lag = net_te_by_lag.groupby("lag_steps").rank(ascending=False).unstack("lag_steps")
rank_corr = rank_by_lag.corr(method="spearman")
rank_corr

Spearman correlation between each pair of lags' zone rankings (1.0 = identical order,
0 = no relationship). High correlation here would support the phase-1 ranking being a real
pattern; low correlation means phase 1's single-lag ranking isn't trustworthy on its own.

## Phase 2b: surrogate significance test

A plug-in TE estimator is biased upward on finite samples (phase 1's noise sanity check
read 0.166, not 0), so no raw TE number can be compared to zero. The standard fix: shuffle
the source series in contiguous blocks (96 bars = 1 day, long enough to preserve each
zone's own short-range autocorrelation structure while destroying its relationship to the
target), recompute TE on the shuffled pair, repeat many times. That builds a null
distribution of "what TE looks like with no real information flow, at this same sample
size and bin occupancy" - a real edge is one whose observed TE clears that null, not one
whose raw value happens to be positive.

Restricted to the 79 borders that exist in the physical graph (158 directed pairs, 200
shuffles each) rather than all 1,806 ordered zone pairs - that's the comparison phase 1's
agreement-rate check actually needs, and keeps this to a couple minutes of
runtime instead of running every pair through 200 shuffles each.

In [ ]:
def block_shuffle(arr: np.ndarray, block_size: int, rng: np.random.Generator) -> np.ndarray:
    n_blocks = len(arr) // block_size
    usable = n_blocks * block_size
    blocks = arr[:usable].reshape(n_blocks, block_size)
    return np.concatenate([blocks[rng.permutation(n_blocks)].reshape(-1), arr[usable:]])


def surrogate_test(
    x: pd.Series,
    y: pd.Series,
    n_bins: int = 8,
    lag: int = 1,
    block_size: int = 96,
    n_shuffles: int = 200,
    rng: np.random.Generator | None = None,
) -> tuple[float, float]:
    rng = rng or np.random.default_rng(0)
    x_bin = pd.qcut(x, n_bins, labels=False, duplicates="drop").to_numpy()
    y_bin = pd.qcut(y, n_bins, labels=False, duplicates="drop").to_numpy()
    real_te = transfer_entropy_from_bins(x_bin, y_bin, lag)
    null_tes = np.array(
        [
            transfer_entropy_from_bins(block_shuffle(x_bin, block_size, rng), y_bin, lag)
            for _ in range(n_shuffles)
        ]
    )
    p_value = (null_tes >= real_te).mean()
    return real_te, p_value


rng = np.random.default_rng(42)
significance_records = []
for zone_a, zone_b in border_pairs:
    te_ab, p_ab = surrogate_test(price_wide[zone_a], price_wide[zone_b], rng=rng)
    te_ba, p_ba = surrogate_test(price_wide[zone_b], price_wide[zone_a], rng=rng)
    significance_records.append(
        {"zone_a": zone_a, "zone_b": zone_b, "te_ab": te_ab, "p_ab": p_ab, "te_ba": te_ba, "p_ba": p_ba}
    )

significance_df = pd.DataFrame(significance_records)
significance_df["a_to_b_significant"] = significance_df["p_ab"] < 0.05
significance_df["b_to_a_significant"] = significance_df["p_ba"] < 0.05
n_sig = (significance_df["a_to_b_significant"] | significance_df["b_to_a_significant"]).sum()
print(f"{n_sig} / {len(significance_df)} borders have at least one significant TE direction (p < 0.05)")

## Phase 2c: agreement rate, restricted to significant edges

Phase 1's 39.2% agreement rate used every border's raw TE comparison, including borders
where both directions sit at or near the noise floor - on those, "which direction is
larger" is close to a coin flip. Recomputing using only borders where at least one
direction clears the surrogate significance test, and where that significant direction is
unambiguous (not both directions significant with similar TE).

In [ ]:
def dominant_significant_direction(row: pd.Series) -> str | None:
    if row["a_to_b_significant"] and not row["b_to_a_significant"]:
        return row["zone_a"]
    if row["b_to_a_significant"] and not row["a_to_b_significant"]:
        return row["zone_b"]
    if row["a_to_b_significant"] and row["b_to_a_significant"]:
        return row["zone_a"] if row["te_ab"] >= row["te_ba"] else row["zone_b"]
    return None


significance_df["te_leader"] = significance_df.apply(dominant_significant_direction, axis=1)
sig_only = significance_df.dropna(subset=["te_leader"]).merge(
    agreement_df.assign(
        zone_a=lambda d: d["border"].str.split(" - ").str[0],
        zone_b=lambda d: d["border"].str.split(" - ").str[1],
    )[["zone_a", "zone_b", "physical_exporter"]],
    on=["zone_a", "zone_b"],
)
sig_only["agrees"] = sig_only["te_leader"] == sig_only["physical_exporter"]
sig_rate = sig_only["agrees"].mean() if len(sig_only) else float("nan")
print(
    f"{sig_only['agrees'].sum()} / {len(sig_only)} borders with a significant TE direction "
    f"agree with physical flow ({sig_rate:.1%})"
    if len(sig_only)
    else "no borders had a significant, unambiguous TE direction at n_shuffles=200"
)

## What phase 2 adds

The lag-sweep correlation above says whether phase 1's price-setting-zone ranking
(`CH`/`RS` at the top) survives looking at 30/60-minute lags instead of just 15 minutes -
read that number before trusting the zone ranking as anything more than a single-lag
artifact.

The surrogate test cuts the 79 borders down to the ones where TE actually clears a
noise-floor baseline specific to that pair's data, and the agreement rate recomputed on
that smaller, cleaner set is the first number in this notebook that's a real finding
rather than an artifact of estimator bias.

Still deferred: a library estimator (`pyinform`/`idtxl`) as a cross-check on the
hand-rolled one, and the cross-check of disagreeing borders against `ci-flows`'s
counter-intuitive-flow events.

## Phase 3: controlling for shared seasonality

Phase 2b's surrogate test came back with 78 of 79 borders "significant" - suspiciously
close to all of them. The tell: `block_shuffle` shuffles whole 96-bar (1-day) blocks, so
a bar's position-within-its-block (its hour of day) survives the shuffle unchanged, and
which day-of-week it lands on is whatever day the block came from, not randomized
separately. Nearly every European zone has the same basic diurnal shape (cheap at night,
expensive at the evening peak) and weekday/weekend gap. That shared shape alone - with no
real cross-zone information flow at all - can make a shuffled source series line up with
a target's real values well enough to look "significant": the surrogate test was
correctly detecting a dependency, just not the one phase 1/2 wanted it to detect.

This section redoes phase 1's ranking, phase 2a's lag sweep, and phase 2b/c's surrogate
test on a deseasonalized residual instead of raw price, and **supersedes** those numbers.
Phase 1/2 above are left as-is, not corrected - they're the worked example of the mistake,
kept for the lesson.

### Deseasonalizing: subtract each zone's own hour-of-day &times; day-of-week mean

168 buckets per zone (24 hours &times; 7 weekdays), additive residual: `residual = price -
mean(price | hour, weekday)`. Matches the confound exactly, since that's the same two
periods `block_shuffle`'s day-aligned blocks preserved. Everything downstream (binning,
`transfer_entropy_from_bins`, `block_shuffle`, `surrogate_test`) is unchanged - only the
input series changes, from raw price to this residual.

In [ ]:
def deseasonalize(price: pd.DataFrame) -> pd.DataFrame:
    hour = price.index.hour * 4 + price.index.minute // 15
    weekday = price.index.weekday
    bucket_mean = price.groupby([hour, weekday]).transform("mean")
    return price - bucket_mean


price_residual = deseasonalize(price_wide)
price_residual.std().describe()

### Re-running phase 1's ranking on the residual

Same net-TE-out-strength ranking as phase 1, same zones, same window - only the series
feeding `transfer_entropy` changed.

In [ ]:
te_records_deseason = []
for source in zones:
    for target in zones:
        if source == target:
            continue
        te = transfer_entropy(price_residual[source], price_residual[target])
        te_records_deseason.append({"source": source, "target": target, "te": te})

te_df_deseason = pd.DataFrame(te_records_deseason)
te_df_deseason.sort_values("te", ascending=False).head(10)

In [ ]:
out_strength_ds = te_df_deseason.groupby("source")["te"].sum()
in_strength_ds = te_df_deseason.groupby("target")["te"].sum()
net_te_deseason = (
    (out_strength_ds - in_strength_ds).rename("net_te_out_strength_deseason").sort_values(ascending=False)
)

fig = px.bar(
    net_te_deseason.rename_axis("zone").reset_index().tail(10),
    x="net_te_out_strength_deseason",
    y="zone",
    orientation="h",
    title=f"Top 10 zones by net TE out-strength, deseasonalized, {date_from} to {date_to}",
)
fig.show()
save_chart(fig, "net_te_out_strength_top10_deseasonalized")

Compare this ranking's top zones against phase 1's `CH`/`RS`-led list above - if the
same zones still lead, that's a real signal surviving the seasonality fix; if the list
changes substantially, phase 1's ranking was mostly the confound.

### Re-running the lag sweep on the residual

In [ ]:
lag_sweep_deseason_records = []
for lag_steps in (1, 2, 4):
    for source in zones:
        for target in zones:
            if source == target:
                continue
            te = transfer_entropy(price_residual[source], price_residual[target], lag=lag_steps)
            lag_sweep_deseason_records.append(
                {"lag_steps": lag_steps, "source": source, "target": target, "te": te}
            )

lag_sweep_deseason_df = pd.DataFrame(lag_sweep_deseason_records)
net_te_by_lag_deseason = (
    lag_sweep_deseason_df.groupby(["lag_steps", "source"])["te"].sum()
    - lag_sweep_deseason_df.groupby(["lag_steps", "target"])["te"].sum()
).rename("net_te_out_strength")

rank_by_lag_deseason = net_te_by_lag_deseason.groupby("lag_steps").rank(ascending=False).unstack("lag_steps")
rank_by_lag_deseason.corr(method="spearman")

### Re-running the surrogate test on the residual

Same 96-bar block shuffle as phase 2b - now defensible, since the series it's shuffling
no longer has the day-shaped signal that made day-aligned blocks dangerous.

In [ ]:
rng_deseason = np.random.default_rng(42)
significance_deseason_records = []
for zone_a, zone_b in border_pairs:
    te_ab, p_ab = surrogate_test(price_residual[zone_a], price_residual[zone_b], rng=rng_deseason)
    te_ba, p_ba = surrogate_test(price_residual[zone_b], price_residual[zone_a], rng=rng_deseason)
    significance_deseason_records.append(
        {"zone_a": zone_a, "zone_b": zone_b, "te_ab": te_ab, "p_ab": p_ab, "te_ba": te_ba, "p_ba": p_ba}
    )

significance_deseason_df = pd.DataFrame(significance_deseason_records)
significance_deseason_df["a_to_b_significant"] = significance_deseason_df["p_ab"] < 0.05
significance_deseason_df["b_to_a_significant"] = significance_deseason_df["p_ba"] < 0.05
n_sig_deseason = (
    significance_deseason_df["a_to_b_significant"] | significance_deseason_df["b_to_a_significant"]
).sum()
print(
    f"{n_sig_deseason} / {len(significance_deseason_df)} borders have at least one "
    f"significant TE direction after deseasonalizing (p < 0.05)"
)

In [ ]:
significance_deseason_df["te_leader"] = significance_deseason_df.apply(dominant_significant_direction, axis=1)
sig_only_deseason = significance_deseason_df.dropna(subset=["te_leader"]).merge(
    agreement_df.assign(
        zone_a=lambda d: d["border"].str.split(" - ").str[0],
        zone_b=lambda d: d["border"].str.split(" - ").str[1],
    )[["zone_a", "zone_b", "physical_exporter"]],
    on=["zone_a", "zone_b"],
)
sig_only_deseason["agrees"] = sig_only_deseason["te_leader"] == sig_only_deseason["physical_exporter"]
sig_rate_deseason = sig_only_deseason["agrees"].mean() if len(sig_only_deseason) else float("nan")
print(
    f"{sig_only_deseason['agrees'].sum()} / {len(sig_only_deseason)} borders with a significant "
    f"deseasonalized TE direction agree with physical flow ({sig_rate_deseason:.1%})"
    if len(sig_only_deseason)
    else "no borders had a significant, unambiguous TE direction after deseasonalizing"
)

## What phase 3 actually tells us

Deseasonalizing resolved one anomaly and falsified the theory behind another.

**Resolved**: the agreement rate was 39.2-39.7% in phases 1-2, below even a coin flip -
on a residual series, it's 51.9%, right at chance. The seasonality confound was
real and was distorting that number.

**Falsified**: phase 2b's theory was that shared diurnal/weekly seasonality explained why
78/79 borders came back "significant." If that were the main driver, removing it should
have cut the significant-border count substantially. It didn't - it went to **79/79**,
every border. Deseasonalizing removed a real confound but didn't touch whatever's making
the surrogate test fire almost everywhere.

**What that leaves us with**: near-universal pairwise statistical dependency between
directly-connected zones, surviving a legitimate fix to the one confound we could name.
The likely explanation is a market that's economically tightly coupled - common shocks
(weather, a generator outage, a demand spike) move prices across much of the system on
the same day, independent of any one border's physical flow direction. That's a
real, citable observation about European day-ahead price integration. It is *not* evidence
of 'borders affecting borders': this surrogate test only covers the 79 zone pairs that
share a physical border, says nothing about non-neighbor pairs, and transfer entropy
doesn't distinguish a direct effect from two zones both reacting to the same cause (the
causality caveat from the top of this notebook applies in full here).

Given that, binary significance isn't a useful filter for this dataset - it clears for
almost every border regardless of bin count, lag, or seasonality handling. The
discriminating information was always in the TE *ranking* (which zone leads, by how
much), not in whether an edge clears a null. `CH`, `RS`, and `MK` leading that ranking,
consistently across phases 1 and 3 and reasonably stable across lags up to 30 minutes
(correlation collapses by 60 minutes), is the one result in this notebook that's survived
every check so far - everything else here is infrastructure and methodology, not yet a
finding about the grid.

Still deferred: a library estimator cross-check on the hand-rolled TE, and the `ci-flows`
counter-intuitive-event cross-check on whichever borders still disagree with physical flow
direction.